In [1]:
# Cell: setup
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

RAW = Path("data/raw/octus")
COMPANY_PATH = RAW / "company_metadata.json"
TRANSCRIPTS_PATH = RAW / "transcripts.json"
SEC_META_PATH = RAW / "sec_filings_metadata.json"
SEC_HTML_DIR = RAW / "sec_html"

assert COMPANY_PATH.exists(), COMPANY_PATH
assert TRANSCRIPTS_PATH.exists(), TRANSCRIPTS_PATH
assert SEC_META_PATH.exists(), SEC_META_PATH
assert SEC_HTML_DIR.exists(), SEC_HTML_DIR

AssertionError: data\raw\octus\company_metadata.json

In [ ]:
# Cell: load json -> dataframes
companies = json.loads(COMPANY_PATH.read_text(encoding="utf-8"))
transcripts = json.loads(TRANSCRIPTS_PATH.read_text(encoding="utf-8"))
sec_meta = json.loads(SEC_META_PATH.read_text(encoding="utf-8"))

companies_df = pd.DataFrame(companies)
transcripts_df = pd.DataFrame(transcripts)
sec_df = pd.DataFrame(sec_meta)

companies_df.head(), transcripts_df.head(), sec_df.head()

In [ ]:
# Cell: parse company_ids and build Octus entity_id -> canonical company map
def parse_company_ids(s: str) -> list[int]:
    if not isinstance(s, str):
        return []
    out = []
    for part in s.split(","):
        part = part.strip()
        if part:
            out.append(int(part))
    return out

companies_df["company_ids_list"] = companies_df["company_ids"].apply(parse_company_ids)

entity_to_company = {}
entity_to_octus_company_id = {}
collisions = {}

for _, row in companies_df.iterrows():
    cname = row["company_name"]
    ocid = row["octus_company_id"]
    for cid in row["company_ids_list"]:
        if cid in entity_to_company and entity_to_company[cid] != cname:
            collisions.setdefault(cid, set()).update([entity_to_company[cid], cname])
        else:
            entity_to_company[cid] = cname
            entity_to_octus_company_id[cid] = ocid

print("Unique entity IDs mapped:", len(entity_to_company))
print("Entity ID collisions (expect 0):", len(collisions))
if collisions:
    print("Example collisions:", list(collisions.items())[:10])

In [ ]:
# Cell: normalize dates (strict parsing)
def parse_transcript_date(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def parse_sec_date(s: str) -> datetime:
    return datetime.strptime(s, "%Y%m%d %H%M%S")

transcripts_df["document_dt"] = transcripts_df["document_date"].map(parse_transcript_date)
sec_df["document_dt"] = sec_df["document_date"].map(parse_sec_date)

print("Transcript date range:", transcripts_df["document_dt"].min(), "->", transcripts_df["document_dt"].max())
print("SEC filing date range:", sec_df["document_dt"].min(), "->", sec_df["document_dt"].max())

In [ ]:
# Cell: audit transcript body lengths (chunking implications)
transcripts_df["body_len"] = transcripts_df["body"].astype(str).str.len()
transcripts_df["body_len"].describe()

In [ ]:
# Cell: map transcripts to company_name via entity_to_company
transcripts_df["company_name"] = transcripts_df["company_id"].map(entity_to_company)
missing_company_name = transcripts_df["company_name"].isna().sum()
print("Transcripts with unmapped company_id:", missing_company_name)
transcripts_df.groupby("company_name").size().sort_values()

In [ ]:
# Cell: map SEC filings to company_name (may have multiple company_ids)
def filing_company_name(company_ids):
    names = {entity_to_company.get(cid) for cid in company_ids if cid in entity_to_company}
    names.discard(None)
    if len(names) == 1:
        return next(iter(names))
    if len(names) == 0:
        return None
    return "|".join(sorted(names))  # ambiguous case

sec_df["company_name"] = sec_df["company_id"].apply(filing_company_name)
print("SEC filings with unmapped company_id list:", sec_df["company_name"].isna().sum())
sec_df.groupby("company_name").size().sort_values()

In [ ]:
# Cell: coverage table — transcripts + filings counts per company_name
transcript_counts = transcripts_df.groupby("company_name").size().rename("transcripts")
filing_counts = sec_df.groupby("company_name").size().rename("sec_filings")

coverage = (
    pd.DataFrame({"company_name": companies_df["company_name"].unique()})
      .merge(transcript_counts, on="company_name", how="left")
      .merge(filing_counts, on="company_name", how="left")
      .fillna(0)
      .astype({"transcripts": int, "sec_filings": int})
      .sort_values(["sec_filings", "transcripts", "company_name"])
)

coverage

In [ ]:
# Cell: SEC HTML presence verification (user confirmed all present; enforce check anyway)
def html_path(doc_id: str) -> Path:
    return SEC_HTML_DIR / f"{doc_id}.html"

sec_df["html_exists"] = sec_df["document_id"].map(lambda x: html_path(x).exists())
missing = sec_df[~sec_df["html_exists"]][["document_id", "document_date", "company_id"]]
print("Missing SEC HTML files (expect 0):", len(missing))
missing.head()

In [ ]:
# Cell: SEC deduplication audit — duplicates by (company_id list, document_type, document_date)
def dedupe_key(row):
    return (tuple(row["company_id"]), row["document_type"], row["document_date"])

sec_df["dedupe_key"] = sec_df.apply(dedupe_key, axis=1)
dup_counts = sec_df["dedupe_key"].value_counts()
dups = dup_counts[dup_counts > 1]

print("Number of duplicate SEC keys:", len(dups))
dups.head(20)